# Distance-tiered evaluation — Safe / Bridge / Wildcard / Random

Genre agreement@10 was flat across every feature variant tried so far (Baseline 1, mfcc-only,
pca-146) — but that metric actively rewards genre conformity, which is the opposite of what
TasteBridge is for. The mission isn't "find same-genre neighbours," it's: recommend tracks that
are familiar, tracks that bridge to something a step away (possibly a different genre/country),
and adventurous tracks that are still connected to the seed rather than random noise.

This notebook builds the evaluation that actually matches that mission: for each seed track, pull
candidates from four distance conditions —

- **Safe** — nearest neighbours by cosine distance (same as Baseline 1's tier)
- **Bridge** — mid-distance band (~45th–55th percentile)
- **Wildcard** — far-distance band (~85th–95th percentile; stops short of the extreme tail to
  avoid degenerate/outlier tracks)
- **Random** — no distance constraint at all, uniformly sampled — the floor. If Wildcard doesn't
  clearly beat Random on human ratings, it isn't a bridge, it's just noise.

**Data-loss note:** neither `baseline1_manual_evaluation.csv` nor `random_manual_evaluation.csv`
currently has ratings on disk (both all-NaN), even though Baseline 1's notebook shows a completed
rating pass in its saved outputs (mean 2.74/5). The csv was most likely overwritten by re-running
the recommendation-generation cell after rating — `data/processed/` isn't git-tracked, so there's
no way to recover the original values. This notebook is a fresh, unified rating pass rather than
an attempt to patch that data back together.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
tracks = pd.read_csv(
    "../data/raw/fma_metadata/tracks.csv",
    index_col=0,
    header=[0, 1]
)

features = pd.read_csv(
    "../data/raw/fma_metadata/features.csv",
    index_col=0,
    header=[0, 1, 2]
)

small_tracks = tracks[
    tracks[("set", "subset")] == "small"
].copy()

small_features = features.loc[small_tracks.index].copy()

X = StandardScaler().fit_transform(small_features.to_numpy())

small_tracks.shape, X.shape

((8000, 52), (8000, 518))

## Build the candidate pool per seed

Same exclusions as every prior baseline: drop the seed itself and any track by the same artist, then rank everything else by cosine distance.

In [3]:
def candidate_pool(seed_id):

    position = small_tracks.index.get_loc(seed_id)
    seed_artist = small_tracks.loc[seed_id, ("artist", "name")]

    similarities = cosine_similarity(
        X[position].reshape(1, -1), X
    )[0]

    pool = pd.DataFrame({
        "track_id": small_tracks.index,
        "artist": small_tracks[("artist", "name")].to_numpy(),
        "title": small_tracks[("track", "title")].to_numpy(),
        "genre": small_tracks[("track", "genre_top")].to_numpy(),
        "similarity": similarities,
    })

    pool = pool[pool["track_id"] != seed_id]
    pool = pool[pool["artist"] != seed_artist]

    return pool.sort_values(
        "similarity", ascending=False
    ).reset_index(drop=True)

## Sample each tier

`safe` = top-3 nearest. `bridge` and `wildcard` are sampled from percentile bands of the ranked pool (evenly spaced within the band, for spread rather than clustering at one edge). `random` ignores the ranking entirely and draws uniformly from the whole valid candidate set. A fixed RNG seed per seed track keeps this reproducible.

In [4]:
def sample_band(pool, low_pct, high_pct, n, rng):

    n_candidates = len(pool)
    start = int(low_pct * n_candidates)
    end = int(high_pct * n_candidates)

    band_positions = np.linspace(start, end - 1, num=n, dtype=int)
    band_positions = np.unique(band_positions)

    return pool.iloc[band_positions]


def sample_tiers_for_seed(seed_id, n_per_tier=3, seed_for_rng=42):

    pool = candidate_pool(seed_id)
    rng = np.random.default_rng(seed_for_rng)

    tiers = {
        "safe": pool.iloc[:n_per_tier],
        "bridge": sample_band(pool, 0.45, 0.55, n_per_tier, rng),
        "wildcard": sample_band(pool, 0.85, 0.95, n_per_tier, rng),
        "random": pool.iloc[
            rng.choice(len(pool), size=n_per_tier, replace=False)
        ],
    }

    rows = []
    for tier_name, subset in tiers.items():
        for _, row in subset.iterrows():
            rows.append({
                "tier": tier_name,
                "recommended_track_id": row["track_id"],
                "recommended_artist": row["artist"],
                "recommended_title": row["title"],
                "recommended_genre": row["genre"],
                "cosine_similarity": row["similarity"],
            })

    return pd.DataFrame(rows)

## Build the full evaluation table

Same 10 fixed seed tracks used in every prior baseline, so this stays comparable.

In [5]:
test_track_ids = [
    140,
    54033,
    59876,
    42234,
    68893,
    43883,
    123978,
    149701,
    63471,
    17608
]

[id_ for id_ in test_track_ids if id_ not in small_tracks.index]

[]

In [6]:
evaluation_rows = []

for seed_id in test_track_ids:

    seed = small_tracks.loc[seed_id]

    tier_rows = sample_tiers_for_seed(
        seed_id,
        n_per_tier=3,
        seed_for_rng=42 + seed_id
    )

    for _, row in tier_rows.iterrows():
        evaluation_rows.append({
            "seed_track_id": seed_id,
            "seed_artist": seed[("artist", "name")],
            "seed_title": seed[("track", "title")],
            "seed_genre": seed[("track", "genre_top")],

            "tier": row["tier"],

            "recommended_track_id": row["recommended_track_id"],
            "recommended_artist": row["recommended_artist"],
            "recommended_title": row["recommended_title"],
            "recommended_genre": row["recommended_genre"],

            "cosine_similarity": row["cosine_similarity"],

            "manual_similarity_1_5": np.nan,
            "connection_type": "",
            "notes": ""
        })

tiered_eval = pd.DataFrame(evaluation_rows)
tiered_eval.shape, tiered_eval["tier"].value_counts()

((120, 13),
 tier
 safe        30
 bridge      30
 wildcard    30
 random      30
 Name: count, dtype: int64)

In [7]:
tiered_eval.to_csv(
    "../data/processed/tiered_manual_evaluation.csv",
    index=False
)

tiered_eval.head(12)

,seed_track_id,seed_artist,seed_title,seed_genre,tier,recommended_track_id,recommended_artist,recommended_title,recommended_genre,cosine_similarity,manual_similarity_1_5,connection_type,notes
0,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,safe,44805,Monk Turner + Fascinoma,Start Again,Pop,0.549527,NaN,,
1,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,safe,6803,Watcha Clan,Balkan Qoulou,International,0.519641,NaN,,
2,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,safe,77564,Bryyn,Abba Ingee,Pop,0.517595,NaN,,
3,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,bridge,59709,Tenores de Aterúe,Ottava Del Tre,International,0.029460,NaN,,
4,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,bridge,85309,Garmisch,They Seem To Be Intelligent (dustmotes Remix),Pop,0.008632,NaN,,
5,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,bridge,118986,Bagat,Mana,Experimental,-0.013071,NaN,,
6,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,wildcard,83613,BOPD,New England Dawn Salendro,Instrumental,-0.155823,NaN,,
7,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,wildcard,105144,Waste Disposal Machine,I Sing the Body Electric (remixed by Torsion),Rock,-0.191258,NaN,,
8,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,wildcard,66535,Waylon Thornton,Baladora,Rock,-0.236925,NaN,,
9,140,Alec K. Redfearn & the Eyesores,Queen Of The Wires,Folk,random,62448,Jahzzar,Julia,Pop,0.171385,NaN,,


## Listen and rate

Same rubric as Baseline 1 (1–5 similarity, `connection_type` tags: rhythm, timbre, mood,
instrumentation, energy, production, harmony, vocals, cross-genre, unclear). Rate blind to
`cosine_similarity` and, importantly, blind to `tier` where possible — knowing a track was sampled
as "wildcard" will bias the rating.

**Do not re-run the cells above once rating starts** — that regenerates `tiered_eval` from scratch
and overwrites the csv, wiping any ratings entered in the meantime (this is what happened to
Baseline 1's data). Rate directly in
`../data/processed/tiered_manual_evaluation.csv` and only re-run the import cell below.

In [8]:
from IPython.display import Audio, display
from pathlib import Path


def get_fma_audio_path(track_id):
    track_id_str = f"{track_id:06d}"
    return (
        Path("../data/raw/fma_small")
        / track_id_str[:3]
        / f"{track_id_str}.mp3"
    )


def listen_to_pair(seed_id, rec_id):
    print("SEED")
    print(small_tracks.loc[seed_id, [("artist", "name"), ("track", "title")]])
    display(Audio(get_fma_audio_path(seed_id)))

    print("\nRECOMMENDATION")
    print(small_tracks.loc[rec_id, [("artist", "name"), ("track", "title")]])
    display(Audio(get_fma_audio_path(rec_id)))

## Leave the audio section above!

In [9]:
# listen_to_pair(seed_id, rec_id)

## Import ratings back in and compare tiers

Run once `manual_similarity_1_5` has been filled in for all 120 rows.

In [10]:
tiered_eval = pd.read_csv(
    "../data/processed/tiered_manual_evaluation.csv"
)

tiered_eval["manual_similarity_1_5"].isna().sum()

np.int64(120)

In [11]:
tier_order = ["random", "wildcard", "bridge", "safe"]

tier_summary = (
    tiered_eval
    .groupby("tier")["manual_similarity_1_5"]
    .agg(
        mean_rating="mean",
        pct_3_or_more=lambda s: (s >= 3).mean(),
        pct_4_or_more=lambda s: (s >= 4).mean(),
        n="count"
    )
    .reindex(tier_order)
)

tier_summary

,mean_rating,pct_3_or_more,pct_4_or_more,n
tier,,,,
random,NaN,0.0,0.0,0
wildcard,NaN,0.0,0.0,0
bridge,NaN,0.0,0.0,0
safe,NaN,0.0,0.0,0


## Reading the result

The core question is whether ratings decay **gracefully** from Safe → Bridge → Wildcard, and
whether Wildcard sits clearly above Random rather than collapsing into it:

- If `wildcard` mean rating ≈ `random` mean rating → the "wildcard" tier isn't actually
  discovering connected-but-distant tracks, it's just far-away noise. The adventurous end of the
  Familiar↔Adventurous slider wouldn't be trustworthy yet.
- If `wildcard` clearly beats `random` (even if well below `safe`) → there's a real signal at
  distance, and a Familiar↔Adventurous control built on this distance metric is justified.
- `bridge` sitting meaningfully above `random` and below `safe` is the shape the whole product
  concept depends on — a smooth, controllable trade-off, not a cliff.

This is the baseline the feature/metric work should now be judged against — not genre_agreement.